# **Vector Spacy**

In [ ]:
import spacy

try:
    nlp = spacy.load("en_core_web_lg")
except OSError:
    print("Downloading spaCy model 'en_core_web_lg'...")
    !python -m spacy download en_core_web_lg
    nlp = spacy.load("en_core_web_lg")

doc = nlp('dog cat banana Soumick')

for token in doc:
  print(token.text, "Vector:", token.has_vector, "OOV : ", token.is_oov)

In [ ]:
doc[0].vector.shape

In [ ]:
base_token = nlp('bread')
base_token.vector.shape

In [ ]:
doc = nlp("bread sandwich burger car tiger human wheat")

for token in doc:
  print(f"{token.text} <-> {base_token.text}:", token.similarity(base_token))

In [ ]:
def print_similarity(base_word, words_to_compare):
  base_token = nlp(base_word)
  doc = nlp(words_to_compare)

  for token in doc:
    print(f"{token.text} <-> {base_token.text}:", token.similarity(base_token))

In [ ]:
print_similarity("iphone", "apple samsung iphone dog kitten")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

king = nlp.vocab['king'].vector
man = nlp.vocab['man'].vector
woman = nlp.vocab['woman'].vector
queen = nlp.vocab['queen'].vector

result = king - man + woman
cosine_similarity([result], [queen])

In [ ]:
import pandas as pd

df1 = pd.read_csv('/content/True.csv', nrows=100)
df1['label'] = df1.apply(lambda row: 1, axis=1)
df2 = pd.read_csv('/content/Fake.csv', nrows=100)
df2['label'] = df2.apply(lambda row: 0, axis=1)

df = pd.concat([df1, df2]).reset_index(drop=True)
df.head()

In [ ]:
nlp = spacy.load("en_core_web_lg")

df['vectors'] = df['text'].apply(lambda text: nlp(text).vector)
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.vectors.values, df.label, test_size=0.2, random_state=2202)

In [ ]:
import numpy as np

X_train_2d = np.stack(X_train)
X_test_2d = np.stack(X_test)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_2d = scaler.fit_transform(X_train_2d)
X_test_2d = scaler.fit_transform(X_test_2d)

model = MultinomialNB()
model.fit(X_train_2d, y_train)

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test_2d)
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train_2d, y_train)

y_pred = model.predict(X_test_2d)
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Truth')

In [ ]:
!pip install gensim
import gensim.downloader as api

wv = api.load('word2vec-google-news-300')
wv.similarity(w1='great', w2='good')

In [ ]:
wv.most_similar('good')

In [ ]:
wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=5)

In [ ]:
wv.doesnt_match(['facebook', 'cat', 'google', 'microsoft'])

In [ ]:
wv.doesnt_match(['dog', 'cat', 'google', 'microsoft'])

In [ ]:
glv = api.load('glove-twitter-25')
glv.most_similar('good')

In [ ]:
glv.most_similar('good')

In [ ]:
glv.doesnt_match('breakfast cereal dinner lunch'.split())

In [ ]:
glv.doesnt_match('facebook cat google microsoft'.split())

# **Fasttext Tutorial**

In [ ]:
!pip install fasttext
import fasttext
import gensim.downloader as api

try:
    model_en = fasttext.load_model('cc.en.300.bin')
    print(model_en.get_nearest_neighbors('good'))
except:
    print('Please upload cc.en.300.bin to the Colab file browser or use a downloader.')

In [ ]:
# Download and decompress the English model (Warning: This is a ~4GB file)
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz
!gunzip cc.en.300.bin.gz

import fasttext
model_en = fasttext.load_model('cc.en.300.bin')
model_en.get_nearest_neighbors('good')

In [ ]:
dir(model_en)

In [ ]:
model_en.get_word_vector('good').shape

In [ ]:
model_en.get_analogies('berlin', 'germany', 'Bangladesh')

In [ ]:
model_en.get_analogies('driving', 'car', 'phone')

In [ ]:
model_en.get_analogies('driving', 'car', 'book')

In [ ]:
model_en.get_nearest_neighbors('chutney')

In [ ]:
model_en.get_nearest_neighbors('halwa')

In [ ]:
model_en.get_nearest_neighbors('saragva', k=5)

In [ ]:
import pandas as pd
df = pd.read_csv('/content/Cleaned_Indian_Food_Dataset.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.TranslatedInstructions[0]

In [ ]:
import re

def preprocess(text):
  text = re.sub(r'[^\w\s]', ' ', text)
  text = re.sub(r'[ \n]+', ' ', text)
  return text.strip().lower()

In [ ]:
df.TranslatedInstructions = df.TranslatedInstructions.apply(preprocess)
df.TranslatedInstructions[0]

In [ ]:
df.to_csv('food_receipes.csv', columns=['TranslatedInstructions'], header=None, index=False)

In [ ]:
model = fasttext.train_unsupervised('/content/food_receipes.csv')

In [ ]:
model.get_nearest_neighbors(preprocess('paneer'))

In [ ]:
model.get_nearest_neighbors('dosa')

In [ ]:
df = pd.read_csv('/content/ecommerceDataset.csv', names=['category', 'description'], header=None)
df.head()

In [ ]:
df.dropna(inplace=True)
df.shape

In [ ]:
df.category.unique()

In [ ]:
df.category.replace('Clothing & Accessories", "Clothing_Accessories', inplace=True)
df.category.unique()

In [ ]:
df['category'] = '__label__' + df['category'].astype(str)
df.head()

In [ ]:
df['category_description'] = df['category'] + ' ' + df['description']
df.head()

In [ ]:
def preprocess(text):
    text = re.sub(r'[^\w\s\']',' ', text)
    text = re.sub(' +', ' ', text)
    return text.strip().lower()

In [ ]:
df['category_description'] = df['category_description'].map(preprocess)
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2)
train.shape, test.shape

In [ ]:
train.to_csv("ecommerce.train", columns=["category_description"], index=False, header=False)
test.to_csv("ecommerce.test", columns=["category_description"], index=False, header=False)

In [ ]:
model = fasttext.train_supervised(input="ecommerce.train")
model.test("ecommerce.test")

In [ ]:
model.predict('ockey mens cotton t shirt fabric details 80 cotton 20 polyestersuper combed cotton rich fabric')

In [ ]:
model.predict("wintech assemble desktop pc cpu 500 gb sata hdd 4 gb ram intel c2d processor 3")